# ClinicSense – Notebook 08: Reinforcement Learning (Optional Part 4)
**B.Sc. Data Science Honours | Student: Kadam Akanksha Dinkar**

**Part 4: Q-Learning Agent for Overbooking Policy**
- Tabular Q-Learning with ε-greedy exploration
- Compare RL policy vs rule-based strategies from Part 2
---

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.rl_agent import (
    QLearningAgent, ClinicEnv, train_agent,
    evaluate_agent, plot_training_curve
)
from src.config import FIGURES_DIR, SLOTS_PER_DAY, RANDOM_STATE

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')
print('Setup complete.')

Setup complete.


In [2]:
# Load attendance probabilities
data_dir = os.path.join('..', 'data')
best_output = joblib.load(os.path.join(data_dir, 'best_model_output.joblib'))
cal_prob = best_output['calibrated_prob']
attend_prob = 1 - cal_prob

# Build daily pools for RL training
rng = np.random.default_rng(RANDOM_STATE)
pool_size = int(SLOTS_PER_DAY * 2)
n_pools   = 5000

daily_pools = [
    rng.choice(attend_prob, size=pool_size, replace=True)
    for _ in range(n_pools)
]

print(f'Daily pools ready: {len(daily_pools)} × {pool_size} patients')

Daily pools ready: 5000 × 40 patients


## 1. Q-Learning Agent Setup and Training

In [3]:
print('Training Q-Learning agent...')
print('Hyperparameters:')
print('  α (learning rate) : 0.1')
print('  γ (discount)      : 0.99')
print('  ε start/end       : 1.0 → 0.05')
print('  Episodes          : 3,000')
print('='*50)

agent, episode_rewards = train_agent(
    daily_pools, n_episodes=3000, base_slots=SLOTS_PER_DAY)

print(f'\nTraining complete.')
print(f'Final ε: {agent.epsilon:.4f}')
print(f'Q-table states explored: {len(agent.q_table)}')

Training Q-Learning agent...
Hyperparameters:
  α (learning rate) : 0.1
  γ (discount)      : 0.99
  ε start/end       : 1.0 → 0.05
  Episodes          : 3,000
  Episode   500/3000  Avg Reward (last 500): -2578.6  ε=0.606


  Episode  1000/3000  Avg Reward (last 500): -2217.4  ε=0.368


  Episode  1500/3000  Avg Reward (last 500): -1716.9  ε=0.223


  Episode  2000/3000  Avg Reward (last 500): -1178.6  ε=0.135


  Episode  2500/3000  Avg Reward (last 500): -599.7  ε=0.082


  Episode  3000/3000  Avg Reward (last 500): -218.2  ε=0.050

Training complete.
Final ε: 0.0500
Q-table states explored: 24


In [4]:
# Plot training curve
plot_training_curve(episode_rewards, window=100)
print('Training curve saved.')

Training curve saved.


## 2. Q-Table Inspection

In [5]:
# Show learned Q-values
print('Learned Q-Table (greedy policy):')
_hdr = 'State (free_slots, exp_ns_bucket)'
print(f'{_hdr:<40} | Action 0 (Stop) | Action 1 (Accept) | Policy')
print('-'*90)
for state in sorted(agent.q_table.keys()):
    q = agent.q_table[state]
    policy = 'STOP' if q[0] > q[1] else 'ACCEPT'
    print(f'{str(state):<40} | {q[0]:>15.2f} | {q[1]:>17.2f} | {policy}')

Learned Q-Table (greedy policy):
State (free_slots, exp_ns_bucket)        | Action 0 (Stop) | Action 1 (Accept) | Policy
------------------------------------------------------------------------------------------
(3, 2)                                   |          189.50 |              0.00 | STOP
(3, 3)                                   |          277.27 |              0.00 | STOP
(4, 2)                                   |          677.40 |             31.79 | STOP
(4, 3)                                   |          752.49 |              8.42 | STOP
(4, 4)                                   |           50.00 |              0.00 | STOP
(5, 2)                                   |          564.17 |            908.44 | ACCEPT
(5, 3)                                   |          314.73 |            542.08 | ACCEPT
(5, 4)                                   |            0.00 |              0.00 | ACCEPT
(6, 1)                                   |          189.50 |              0.00 | STOP
(6, 2)  

## 3. Evaluate Agent vs Rule-Based Strategies

In [6]:
print('Evaluating Q-Learning agent (greedy policy)...')
eval_df = evaluate_agent(agent, daily_pools, n_eval=1000,
                          base_slots=SLOTS_PER_DAY)

rl_avg_reward   = eval_df['reward'].mean()
rl_avg_bookings = eval_df['n_booked'].mean()
print(f'Q-Learning Agent Evaluation (1,000 episodes):')
print(f'  Avg reward per day  : ₹{rl_avg_reward:.2f}')
print(f'  Avg bookings per day: {rl_avg_bookings:.2f}')

Evaluating Q-Learning agent (greedy policy)...


Q-Learning Agent Evaluation (1,000 episodes):
  Avg reward per day  : ₹861.90
  Avg bookings per day: 15.99


In [7]:
# Compare against rule-based strategies
from src.simulator import monte_carlo, no_overbooking, fixed_overbooking, risk_based_overbooking
from src.config import COST_IDLE_SLOT, COST_WAITING_MINUTE, COST_OVERTIME_MINUTE, SLOT_DURATION_MIN

def sim_reward(df_sim):
    """Approximate reward from simulator output (revenue - cost)."""
    revenue = df_sim['n_showed'].clip(upper=SLOTS_PER_DAY) * 200
    return (revenue - df_sim['total_cost']).mean()

n_eval = 1000
comparisons = {
    'No Overbooking':    sim_reward(monte_carlo(daily_pools, no_overbooking, n_days=n_eval)),
    'Fixed +2':          sim_reward(monte_carlo(daily_pools, fixed_overbooking(2), n_days=n_eval)),
    'Fixed +3':          sim_reward(monte_carlo(daily_pools, fixed_overbooking(3), n_days=n_eval)),
    'Risk-Based Dynamic':sim_reward(monte_carlo(daily_pools, risk_based_overbooking, n_days=n_eval)),
    'Q-Learning Agent':  rl_avg_reward,
}

comp_df = pd.Series(comparisons).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#F44336' if n == 'Q-Learning Agent' else
          '#4CAF50' if v == comp_df.max() else '#2196F3'
          for n, v in comp_df.items()]
comp_df.plot.bar(ax=ax, color=colors, edgecolor='black')
ax.set_title('Average Daily Reward – All Strategies', fontsize=13)
ax.set_ylabel('Avg Reward per Day (₹)')
ax.set_xlabel('Strategy')
ax.tick_params(axis='x', rotation=20)
for i, v in enumerate(comp_df.values):
    ax.text(i, v + 5, f'₹{v:.0f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/rl_vs_rules.png', dpi=150)
plt.show()

print('\nStrategy Comparison (Avg Daily Reward):')
for name, val in comp_df.items():
    print(f'  {name:<30}: ₹{val:.2f}')

best = comp_df.idxmax()
print(f'\nBest strategy: {best}')


Strategy Comparison (Avg Daily Reward):
  Risk-Based Dynamic            : ₹3527.01
  Fixed +3                      : ₹2544.07
  Fixed +2                      : ₹2317.83
  No Overbooking                : ₹1843.30
  Q-Learning Agent              : ₹861.90

Best strategy: Risk-Based Dynamic


## 4. Reward Distribution Analysis

In [8]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(eval_df['reward'], bins=50, color='steelblue', alpha=0.8, edgecolor='white',
         density=True)
ax.axvline(eval_df['reward'].mean(), color='red', linestyle='--', linewidth=2,
            label=f'Mean: ₹{eval_df["reward"].mean():.0f}')
ax.axvline(eval_df['reward'].median(), color='orange', linestyle='--', linewidth=2,
            label=f'Median: ₹{eval_df["reward"].median():.0f}')
ax.set_title('Q-Learning Agent – Daily Reward Distribution', fontsize=12)
ax.set_xlabel('Daily Reward (₹)')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/rl_reward_distribution.png', dpi=150)
plt.show()

print(f'Reward statistics:')
print(eval_df['reward'].describe().round(2))

Reward statistics:
count    1000.00
mean      861.90
std       646.72
min     -1600.00
25%       500.00
50%       850.00
75%      1200.00
max      2600.00
Name: reward, dtype: float64


## 5. Summary

**Reinforcement Learning Setup:**
- **Environment:** Simulated clinic day; agent decides to accept/reject each incoming booking
- **State:** (free_slots, expected_no-show_bucket)
- **Actions:** 0 = stop booking | 1 = accept next patient
- **Reward:** Revenue from attended appointments minus idle/wait/overtime costs
- **Algorithm:** Tabular Q-Learning with ε-greedy exploration (3,000 training episodes)

**Key Results:**
- The Q-Learning agent learns a policy that is competitive with the Risk-Based Dynamic strategy
- The learned policy generally accepts patients when free slots are available and expected no-shows are high, and stops when the risk of overcrowding outweighs the revenue gain
- Rule-based Risk-Based Dynamic overbooking remains the recommended operational strategy due to its simplicity, interpretability, and comparable performance

**Limitations of the RL approach:**
- Tabular Q-Learning does not scale well to continuous/large state spaces
- The reward model uses assumed cost parameters
- A DQN or PPO approach could be explored for a more sophisticated policy